Economies of scale: how much of poverty is household size?
==========================================================

**Author:** Ethan Ligon



## What this is



Session 2 wrote down $c_i = C_i / A_i^\theta$, said that $\theta$ is
assumed rather than estimated, and told you to report poverty for several
values of it.  This notebook does that, and then asks the question that
makes the choice matter: as $\theta$ moves, does the *ranking* of Ghana's
regions move with it?

The answer turns out to depend on something session 2 didn't say, which is
what you hold fixed while $\theta$ varies.  Hold the line fixed and the
national headcount falls by two thirds between $\theta = 1$ and
$\theta = 0.5$, for a reason that has nothing to do with anyone's welfare.
Hold the headcount fixed and the ranking barely moves.  Both are shown
below; deciding which one you'd publish is the exercise.

-   **Prerequisites:** `lsms_library` on the release kernel, GhanaLSS microdata.
    Self-contained; it doesn't assume `session2.ipynb` has run.



## Setup



Session 2's aggregate, weights and FGT function, restated so this notebook
stands on its own.  `food_total` here is food *purchases* per household: own
production is left out to keep the notebook short.  Session 2's aggregate
includes it, and exercise 1 at the end asks you to.



In [1]:
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ghana = ll.Country('GhanaLSS')
wave = '2016-17'

food_total = ghana.food_expenditures().groupby(['t', 'i']).sum().squeeze()
sample = ghana.sample()

C = food_total.xs(wave, level='t')                 # cedi per household, purchases
s = sample.xs(wave, level='t').reindex(C.index)
w, region = s.weight, s.strata                     # strata carries the region name

def fgt(c, z, alpha=0, w=1):
    """
    Weighted FGT_alpha.  
    c: consumption per adult equivalent; 
    z: poverty line; 
    w: weights.

    Sums over the poor only: see session 3 for why.
    """
    # A scalar broadcasts and a Series reindexes, so `w=1' (unweighted) and a
    # real weight column take the same path, aligned on the index.
    w = pd.Series(w, index=c.index, dtype=float)
    # Infinities are not missing values to pandas, so say so before dropping.
    df = (pd.concat({'c': c, 'w': w}, axis=1)
            .replace([np.inf, -np.inf], np.nan)
            .dropna())
    poor = df.c < z
    gap = (z - df.c[poor]) / z
    return (df.w[poor] * gap ** alpha).sum() / df.w.sum()

print(f"{len(C)} households in {wave}; regions: {sorted(region.unique())}")

`strata` is the sampling stratum, and in GLSS7 that is the region: the ten
regions Ghana had before the 2019 split, so "Northern" here is the old
Northern Region, Savannah and North East included.  When this notebook says
*north* it means Northern, Upper East and Upper West.



## 1.  Adult equivalents



`household_characteristics()` gives counts by age-sex cell.  An adult
equivalent scale is a set of weights over those cells, and this one is
deliberately crude: anyone under 14 counts as `CHILD` adults, anyone 14 or
over counts as one.  It's a parameter because it should be.



In [1]:
hc = ghana.household_characteristics().xs(wave, level='t').droplevel('v')
cells    = [c for c in hc.columns if c[:2] in ('F ', 'M ')]
children = [c for c in cells if c.split()[1] in ('00-03', '04-08', '09-13')]
adults   = [c for c in cells if c not in children]

CHILD = 0.5                                        # an adult equivalent per child

n = hc[cells].sum(axis=1).reindex(C.index)         # persons
A = (hc[adults].sum(axis=1) + CHILD * hc[children].sum(axis=1)).reindex(C.index)

pd.DataFrame({'persons': n, 'adult equiv': A, 'ratio': A / n}).describe().round(2)

The median household has four people and three adult equivalents.  Nineteen
households in the expenditure table have no roster row, and they drop out
of everything below; `fgt` discards non-finite values, so they drop silently.
Check that you're comfortable with that before you publish anything.

**Exercise 1.1.** The ratio `A/n` runs from 0.5 to 1.  Tabulate its mean by
region.  Which regions have the most children per adult, and what does that
already tell you about the direction the equivalence adjustment will push
the regional ranking?



## 1.  The headcount as a function of theta, holding the line fixed



Take the line to be the (unweighted) lower quartile of consumption per adult
equivalent at $\theta = 1$, the direct analogue of session 2's placeholder.
Then hold it fixed and lower $\theta$.



In [1]:
def c_of(theta):
    return C / A ** theta

z = c_of(1.0).quantile(0.25)                       # cedi per adult equivalent
thetas = np.round(np.linspace(0.5, 1.0, 11), 2)

P0 = pd.Series({th: fgt(c_of(th), z, w=w) for th in thetas}, name='P0')
print(f"line z = {z:.2f} per adult equivalent")
print(P0.round(4).to_string())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(P0.index, P0.values, marker='o', color='#1f4e79')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel('weighted headcount ratio')
ax.set_title(f'National headcount at a fixed line, z = {z:.0f} per adult equivalent')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

The headcount falls from 0.137 at $\theta = 1$ to 0.041 at
$\theta = 0.5$, and the reason is arithmetic, not welfare.
$A_i^\theta \le A_i$ for every household with more than one adult
equivalent, so lowering $\theta$ raises everyone's $c_i$ except the
single-adult households, while the line stays where it was.  A fixed line
in "per adult equivalent" units means something different at each
$\theta$, and this figure is mostly a picture of that.

**Exercise 2.1.** Who leaves poverty as $\theta$ falls?  Compute the mean
household size of the poor at $\theta = 1, 0.75, 0.5$.  Lanjouw and
Ravallion (1995) called the result "poverty and household size"; state it in
one sentence from your numbers.



## 1.  The regional ranking



The question session 2 actually asked.  Same fixed line, headcount by
region, and the rank of each region among the ten.



In [1]:
def by_region(c, w, z):
    df = pd.DataFrame({'c': c, 'w': w, 'r': region})
    return df.groupby('r').apply(lambda d: fgt(d.c, z, w=d.w))

H = pd.DataFrame({th: by_region(c_of(th), w, z) for th in thetas})
ranks = H.rank(ascending=False).astype(int)        # 1 = poorest

moved = (ranks.diff(axis=1) != 0).iloc[:, 1:].any()
print("regional headcount (rows) by theta (columns):")
print(H.round(3).to_string())
print("\nranks change between adjacent thetas at:", moved[moved].index.tolist())
print(ranks.loc[:, [0.5, 0.85, 0.9, 1.0]].to_string())

Over $[0.5, 0.80]$ nothing moves: Upper West, Upper East and Northern are
the three poorest regions in that order at every $\theta$, and Greater
Accra is last.  Between 0.80 and 0.85 Central and Eastern swap in the
middle; between 0.85 and 0.90 Upper East overtakes Upper West at the top
and Brong Ahafo and Volta trade places, then trade back at 0.95.  The
ranking is stable for most of the range and unstable exactly where two
regions are close, which is the top and the middle.



In [1]:
north = ['Northern', 'Upper East', 'Upper West']
colors = {'Upper West': '#8c2d04', 'Upper East': '#d95f0e', 'Northern': '#fe9929'}

fig, ax = plt.subplots(figsize=(6.5, 4))
for r in H.index:
    if r in north:
        ax.plot(H.columns, H.loc[r], color=colors[r], lw=2)
        ax.annotate(r, (H.columns[-1], H.loc[r].iloc[-1]),
                    xytext=(4, 0), textcoords='offset points', va='center', color=colors[r])
    else:
        ax.plot(H.columns, H.loc[r], color='#b0b0b0', lw=1)
ax.set_xlabel(r'$\theta$'); ax.set_ylabel('weighted headcount ratio')
ax.set_title('Regional headcount at a fixed line; the north in colour, the south in grey')
ax.set_xlim(0.5, 1.12)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

**Exercise 3.1.** The two northernmost regions cross.  Say why in terms of
their household sizes: pull the mean of `A` for Upper East and Upper West
and show which one lowering $\theta$ favours.



## 1.  Holding the headcount fixed instead



The alternative is to let the line move with $\theta$, so that it is
always the lower quartile of whatever $c_i$ currently is.  The national
headcount is then 0.25 unweighted by construction, and what's left of
$\theta$ is purely who is poor, not how many.



In [1]:
Hrel = pd.DataFrame({th: by_region(c_of(th), w, c_of(th).quantile(0.25)) for th in thetas})
ranks_rel = Hrel.rank(ascending=False).astype(int)
moved_rel = (ranks_rel.diff(axis=1) != 0).iloc[:, 1:].any()

print("relative line: ranks change between adjacent thetas at:", moved_rel[moved_rel].index.tolist())
print(pd.concat({'headcount': Hrel[[0.5, 0.7, 1.0]].round(3),
                 'rank': ranks_rel[[0.5, 0.7, 1.0]]}, axis=1).to_string())

With a relative line, only one pair ever changes places: Upper West is
poorest up to $\theta = 0.65$ and Upper East from 0.70 on, by margins of
a point or two.  Everything else is fixed from 0.5 to 1.  So the answer to
session 2's question is "hardly at all, and only at the top", once the
mechanical effect of the line is taken out.

**Exercise 4.1.** Repeat section 3 with `CHILD` at 0.3 and at 1.0 (the
per-capita case).  Does the child weight or $\theta$ do more to the
ranking?  Report the range of the Upper East headcount across all six
combinations.

**Exercise 4.2.** Neither section 2 nor section 4 is what a statistical office
does.  The usual practice is to fix the line in per-adult-equivalent terms
at $\theta = 1$ and never vary $\theta$ at all.  Write two sentences
defending that, using the figure in section 2, and one sentence saying what
it hides.



## Exercises



1.  Everything above is on food purchases.  Add own production, valued as in
    the `food_sources` notebook, and redo section 3.  Own production is a
    larger share in the north, and northern households are larger, so the
    two adjustments push in the same direction; say whether the ranking is
    more or less stable once both are in.
2.  Session 2's Barten slide argued that a single $A^\theta$ makes a child
    a fraction of an adult *for every good*.  Test the simplest implication:
    split food expenditure into cereals and everything else and compute the
    cereal budget share by `A/n`.  If children re-price goods rather than
    scale the bundle, the share should move with composition at fixed
    $c_i$.  Does it?
3.  The sample weights in `sample()` are household weights.  A headcount
    ratio is a share of *people*.  Reweight by `weight * n` and redo section
    
    1.  How much of the fixed-line decline in the headcount is the weighting
    
    convention rather than $\theta$?

